# Iterators, Maps, and Intro to Inheritance

### Vector vs List Indexing; `auto`; Iterators; Range-based `for`; Inheritance: Account, Savings, Checking; More Inheritance Practice; `std::find` and the STL Algorithms; Maps; Memoized Fibonacci with `static`; Where to Go From Here.

<br><br><br><br><br>
<br><br><br><br><br>

# 1. Vector vs List Indexing

### We've been building and using linked lists, and we introduced the `std::list`. Today we'll look at how you loop over a `std::list`. The starting point is a question about indexing.

### We already know `std::vector<int>` supports `v[i]`. What about `std::list<int>`?

In [ ]:
////vectorVsList.cpp
// Example 1a - Predict, then run! Does this compile?
//%CXXFLAGS -std=c++11 
// c++11 needed for initializer lists
#include <iostream>
#include <vector>
#include <list>

int main(){
    std::vector<int> v = {10, 20, 30, 40};
    std::list<int>   l = {10, 20, 30, 40};

    std::cout << v[2] << "\n";          
    // std::cout << l[2] << "\n";      
    return 0;
}

### **Why does `std::list` refuse?** `std::vector` stores its elements in one contiguous block of memory. `v[2]` is one arithmetic step: *base address + 2 × element size*. Same cost for `v[2]` and `v[2000000]`. We call that **constant time**, or **O(1)**.

### `std::list` is a chain of nodes scattered across the heap, connected by pointers. To get to position 2 you'd follow pointers twice. To get to position 2000, 2000 times. The cost grows with the position. We call that **linear time**, or **O(n)** (specifically O(i) for position `i`).

### The STL refused to give `std::list` an `operator[]` because they didn't want the syntax to *lie* about the cost. Instead, `std::list` exposes the walking step explicitly, through **iterators**.

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 2. `auto`

 ### `auto` tells the compiler *"figure out the type from the right-hand side."* The type is still fixed at compile time.

In [ ]:
////autoDemo.cpp
// Example 2a - auto deduces the type from the initializer
//%CXXFLAGS -std=c++11
#include <iostream>
#include <string>

int main(){
    auto x = 5;             
    auto y = 5.0;           
    auto s = "hi";  // actually a C-style string (const char*); the older, more primitive way C++ inherited from C
    
    std::cout << x << " " << y << " " << s << "\n";   
   
    return 0;
}

### `auto` is for when writing the type out hurts more than it helps, and this will come up mostly with iterators that we are about to see. When in doubt, write the type.


<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 3. Iterators

### An **iterator** is an object that behaves like a pointer into a container. For `std::list`, an iterator wraps a `Node*` internally and exposes three pointer-like operations:

| Operation | What it does |
|---|---|
| `*it` | read the current element |
| `++it` | move to the next element |
| `it != other` | compare with another iterator (especially `end()`) |

### Two methods give you iterators to start and stop with:

- ### `lst.begin()` is an iterator at the **first** element.
- ### `lst.end()` is an iterator **one past the last** element. Not a real element, just a sentinel meaning "off the end."

### Why one past? So that the same loop pattern works on every list, *including the empty one*. `it != lst.end()` is the natural stopping condition.

In [ ]:
////iterFirst.cpp
// Example 3a - iterators on std::list, by hand
//%CXXFLAGS -std=c++11
#include <iostream>
#include <list>

int main(){
    std::list<int> lst = {10, 20, 30, 40};

    std::list<int>::iterator it = lst.begin();   
    std::cout << *it << "\n";                    
    ++it;
    std::cout << *it << "\n";                    
    ++it;
    std::cout << *it << "\n";                   
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>

### Notice the type `std::list<int>::iterator`. That's the long type `auto` was made for. Most of the time we'll just write `auto it = lst.begin();` and let the compiler fill it in.

In [ ]:
////iterLoop.cpp
// Example 3b - the full for-loop pattern, with auto
//%CXXFLAGS -std=c++11
#include <iostream>
#include <list>
#include <vector>

int main(){
    std::list<int>   lst = {10, 20, 30, 40};
    std::vector<int> vec = {1, 2, 3, 4, 5};

    for (auto it = lst.begin(); it != lst.end(); ++it){
        std::cout << *it << " ";
    }
    std::cout << "\n";                          

    // The exact same pattern works on vector. Iterators are the common interface.
    for (auto it = vec.begin(); it != vec.end(); ++it){
        std::cout << *it << " ";
    }
    std::cout << "\n";                         
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>

### Same loop shape, two different containers. That's the payoff of iterators: every STL container offers `begin()`/`end()`, so once you know the pattern you can walk anything.

### **You try!** (Example 3c) Write a function `sumList` that takes a `std::list<int>` and returns the sum of its elements, using iterators. Test it on `{10, 20, 30, 40}`. It might be a good idea to pass the list as a const reference (why?).

In [ ]:
////sumList.cpp
// Example 3c - sumList with iterators
// (You write this!)
//%CXXFLAGS -std=c++11
#include <iostream>
#include <list>

int main(){
    // TODO: write your solution here, then test it.
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 4. Range-based `for`

### The `for (auto it = ...; it != ...; ++it)` pattern is verbose for something so common. C++ gives us syntactic sugar for it: the **range-based for loop**.

```cpp
for (int x : lst) { ... }
```

### Read it as *"for each `x` in `lst`."* Under the hood, the compiler rewrites it into the same `begin()`/`end()`/`++` loop we just wrote. So it works on *any* container that provides `begin()` and `end()`.

In [ ]:
////rangeFor.cpp
// Example 4a - same loop, two styles, two containers
//%CXXFLAGS -std=c++11
#include <iostream>
#include <list>
#include <vector>

int main(){
    std::list<int>   lst = {10, 20, 30, 40};
    std::vector<int> vec = {1, 2, 3, 4, 5};

    // Range-based for - copies each element into x
    for (int x : lst){
        std::cout << x << " ";
        x *= 10;
    }
    std::cout << "\n";    
    
    for (int x : lst){
        std::cout << x << " ";
    }
    std::cout << "\n"; 


    // int& binds x to the actual element
    for (int& x : lst){
        x *= 10;
    }
    
    for (int x : lst){
        std::cout << x << " ";
    }
    std::cout << "\n"; 

    // Note: each lst could be replaced with vec!
    
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>

### Two things to notice:

### 1.  **`for (int x : lst)`** copies each element into `x`. Cheap for `int`, but for big types (e.g., `std::string`) prefer `for (const auto& x : lst)` to avoid the copy.
###  2. **`for (int& x : lst)`** binds `x` *by reference* to the real element. Writing to `x` writes to the container. That's how the second loop multiplied every element by 10.

### **You try!** (Example 4b) Write a function `countOccurrences` that takes a `std::list<int>` and a value, and returns how many times the value appears in the list. Use a range-based `for` loop. Test it on `{1, 2, 3, 2, 4, 2, 5}` looking for `2`. You should get `3`.

In [ ]:
////countOccurrences.cpp
// Example 4b - countOccurrences with range-based for
// (You write this!)
//%CXXFLAGS -std=c++11
#include <iostream>
#include <list>

int main(){
    // TODO: write your solution here, then test it.
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 5. Inheritance: Account, Savings, Checking

### A note on scope. Inheritance is a big topic. Today we're covering the **basics**: factoring shared code into a parent class, accessing parent data from a child class, and calling parent constructors. We're **not** covering:

- ### multiple inheritance (one class inheriting from more than one parent)
- ### virtual functions and polymorphism (picking the right method at runtime based on the actual object type)
- ### abstract classes and pure virtual functions

### You'll see all of those if you do more C++ programming. For now, the goal is to be able to read inheritance code and write simple hierarchies of your own.

### **Motivation.** Imagine you're writing two classes for a bank: `SavingsAccount` and `CheckingAccount`. Both need a balance, an owner name, a `deposit()` method, and a `withdraw()` method. They also each have something unique: savings has an interest rate, checking has an overdraft limit. Without inheritance, the shared stuff has to be written twice.

In [ ]:
////accountsNoInheritance.cpp
// Example 5a - Two classes with duplicated code. Not great.
#include <iostream>
#include <string>

class SavingsAccount {
private:
    std::string owner;
    double balance;
    double interestRate;
public:
    SavingsAccount(std::string o, double b, double r)
        : owner(o), balance(b), interestRate(r) {}
    void deposit(double amount){ balance += amount; }
    void withdraw(double amount){ balance -= amount; }
    void displayInfo() const {
        std::cout << owner << " (savings) balance: " << balance
                  << ", rate: " << interestRate << "\n";
    }
};

class CheckingAccount {
private:
    std::string owner;
    double balance;
    double overdraftLimit;
public:
    CheckingAccount(std::string o, double b, double od)
        : owner(o), balance(b), overdraftLimit(od) {}
    void deposit(double amount){ balance += amount; }       
    void withdraw(double amount){ balance -= amount; }     
    void displayInfo() const {                             
        std::cout << owner << " (checking) balance: " << balance
                  << ", overdraft: " << overdraftLimit << "\n";
    }
};

int main(){
    SavingsAccount  s("Alice", 1000, 0.05);
    CheckingAccount c("Bob",   500, 200);
    s.displayInfo();
    c.displayInfo();
    return 0;
}

### Notice the duplication. `deposit`, `withdraw`, and most of `displayInfo` are written twice. If we ever change how a deposit works (say, add a logging line), we have to change it in two places. With more account types (Money Market, Business, ...) it gets worse.

### **Inheritance** lets us pull the shared stuff into a **base class** (`Account`) and have **derived classes** (`SavingsAccount`, `CheckingAccount`) automatically get those members.
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

### Three pieces of syntax to learn:

### 1. **`class Derived : public Base { ... }`** declares that `Derived` inherits from `Base`. The `public` keyword means: public members of `Base` stay public in `Derived` (most common choice).
### 2. **`protected`** is a third access level alongside `public` and `private`. A `protected` member is private to outside code but accessible to derived classes. We mark `balance` as `protected` so derived classes can read and write it.
### 3. **Constructor chaining.** When a derived object is constructed, the base class's constructor runs first. To pass arguments to it, we use the same initializer-list syntax we use for member initialization: `SavingsAccount(...) : Account(o, b) { ... }`.

In [ ]:
////accountsInheritance.cpp
// Example 5b - Same thing, with inheritance. Shared code lives in Account.
#include <iostream>
#include <string>

class Account {
protected:                              // accessible to derived classes
    std::string owner;
    double balance;
public:
    Account(std::string o, double b) : owner(o), balance(b) {}
    void deposit(double amount){ balance += amount; }
    void withdraw(double amount){ balance -= amount; }
    void displayInfo() const {
        std::cout << owner << " balance: " << balance << "\n";
    }
};

class SavingsAccount : public Account {
private:
    double interestRate;
public:
    SavingsAccount(std::string o, double b, double r)
        : Account(o, b), interestRate(r) {}     // chain to base constructor

    void addInterest(){
        balance += balance * interestRate;      // can access balance: it's protected
    }
};

class CheckingAccount : public Account {
private:
    double overdraftLimit;
public:
    CheckingAccount(std::string o, double b, double od)
        : Account(o, b), overdraftLimit(od) {}
};

int main(){
    SavingsAccount  s("Alice", 1000, 0.05);
    CheckingAccount c("Bob",   500, 200);

    s.deposit(200);          
    s.addInterest();         
    s.displayInfo();         

    c.withdraw(100);         
    c.displayInfo();         
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>

### Both `SavingsAccount` and `CheckingAccount` automatically have `deposit`, `withdraw`, and `displayInfo`. We only wrote those methods once, in `Account`.

### A few things to highlight:

- ### The line `class SavingsAccount : public Account` is what says "is-a Account."
- ### `: Account(o, b)` in the constructor passes `owner` and `balance` up to the base constructor. The base constructor runs first, then the derived constructor body.
- ### `balance` is `protected` in `Account`, which is why `SavingsAccount::addInterest()` can read and write it. If we'd made it `private`, this wouldn't compile.

### **One-sentence guidance on when to use inheritance.**

> **is-a → inheritance, has-a → composition.**

### A `SavingsAccount` *is-a* `Account`, so inheritance fits. A `Car` *has-a* `Engine`, so composition fits (give `Car` an `Engine` member, don't inherit from it). When in doubt, ask which relationship you mean.

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 6. More Inheritance - Practice


### **Exercise 6a — Shape.** Write a base class `Shape` that stores a `color` (string), has a constructor that sets it, and has a method `printColor()` that prints the color. Then write two derived classes. `Circle` adds a `radius` and a method `area()` that returns `3.14159265 * radius * radius`. `Rectangle` adds a `width` and a `height` and a method `area()` that returns `width * height`. Make a `Circle` and a `Rectangle`, call `printColor()` on each (inherited from `Shape`), and call `area()` on each (its own method).

In [1]:
////shape.cpp
// Example 6a - Shape with Circle and Rectangle
// (You write this!)
#include <iostream>
#include <string>

class Shape{
protected:
    std::string color;
public:
    Shape(std::string c) : color(c) {}
    void printColor() const{
        std::cout << "Color: " << color << std::endl;
    }

};
class Circle : public Shape {
    private:
        double radius;
    public:
        Circle(std::string c, double r) : Shape(c), radius(r) {}
        double area() const{
            return 3.14159265 * radius * radius;
        }
};

class Rectangle :  public Shape{
    private:
        double width;
        double height;
    public:
        Rectangle(std::string c, double w,  double h) : Shape(c), width(w), height(h){} 
        double area(){
            return width * height;
        }

};

int main(){
    // TODO: write your solution here, then test it.
    Circle circle("red", 3.0);
    Rectangle rectangle("green", 4.0, 3.0);
    circle.printColor();
    std::cout << "Circle area: " << circle.area() << std::endl;
    
    rectangle.printColor();
    std::cout << "rectangle area: " << rectangle.area() << std::endl ;
    return 0;
}

wrote file shape.cpp
$> g++   shape.cpp    -o shape
$> ./shape 
Color: red
Circle area: 28.2743
Color: green
rectangle area: 12



<br><br><br><br><br>
<br><br><br><br><br>

### **Exercise 6b — Animal.** Write a base class `Animal` with a `name` (string), a constructor that sets the name, and a method `introduce()` that prints `"Hi, I am <name>."`. Then write `Dog` and `Cat`, each adding its own method `speak()`: `Dog::speak()` prints `"Woof!"` and `Cat::speak()` prints `"Meow!"`. Make a `Dog` and a `Cat`, call `introduce()` on each (inherited from `Animal`), and call `speak()` on each (its own method).

In [ ]:
////animal.cpp
// Example 6b - Animal with Dog and Cat
// (You write this!)
#include <iostream>
#include <string>

class Animal{
    protected:
        std::string name;
    public:
        Animal(std::string n) : name(n) {}
        void introduce(){ 
            std::cout << "Hi I am"
        }
        
};

class Dog : public Animal{
    private:
    public:

};
class Cat : public Animal{
    private:
    public:

};
int main(){
    // TODO: write your solution here, then test it.
    
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 7. `std::find` and the STL Algorithms

### Iterators aren't just for writing your own loops. They're the **interface** the entire `<algorithm>` library is built on. The simplest example is `std::find`:

```cpp
auto it = std::find(lst.begin(), lst.end(), value);
```

### `std::find` walks from `begin` to `end`. It returns an iterator to the first match, or returns `end` if nothing matches. That's why the "not found" check is `it == lst.end()`.

In [ ]:
////findDemo.cpp
// Example 7a - std::find on list and vector
//%CXXFLAGS -std=c++11
#include <iostream>
#include <list>
#include <vector>
#include <algorithm>

int main(){
    std::list<int>   lst = {10, 20, 30, 40};
    std::vector<int> vec = {1, 2, 3, 2, 4, 2, 5};

    auto it1 = std::find(lst.begin(), lst.end(), 30);
    if (it1 != lst.end())
        std::cout << "found " << *it1 << " in lst\n";   
    else
        std::cout << "not in lst\n";

    auto it2 = std::find(vec.begin(), vec.end(), 99);
    if (it2 != vec.end())
        std::cout << "found " << *it2 << " in vec\n";
    else
        std::cout << "not in vec\n";    
        
    
    std::cout << std::count(vec.begin(), vec.end(), 2) << "\n"; // what do you think this does?

    return 0;
}


### Same function, both containers. That's the whole pitch for iterators. Other algorithms in `<algorithm>` work the same way: e.g. `std::sort` (on random-access iterators like `vector`'s), and dozens more, all written against the iterator interface.


<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 8. Maps

### You've used Python dictionaries: `eng_spn = {'dog': 'perro', 'cat': 'gato'}`, then read a value with `eng_spn['dog']`. C++ has the same idea, called a **map**: a container of **key, value** pairs that you look up *by key*, not by position. The one new thing in C++ is that you write the key type and the value type when you create the map. Python decides those types automatically. C++ requires you to state them.

### The version you'll use most often is `std::unordered_map<Key, Value>`, in `<unordered_map>`. It's a hash table, the same kind of data structure Python uses for its `dict`. There's also `std::map<Key, Value>` (in `<map>`) that keeps its keys in sorted order. We'll use `std::unordered_map` here and explain the difference at the end of the section.

### Real-world analogies:
- ### a phone book: name (key) maps to phone number (value)
- ### a translation dictionary: English word (key) maps to Spanish word (value)
- ### a word counter: word (key) maps to count (value)

In [ ]:
////mapBasics.cpp
// Example 8a - A map of keys to values, like a Python dict
//%CXXFLAGS -std=c++11
#include <iostream>
#include <unordered_map>
#include <string>

int main(){
    // In Python: eng_spn = {'dog':'perro', 'cat':'gato', 'horse':'caballo'}
    // In C++ you write the key type and the value type: <std::string, std::string>.
    std::unordered_map<std::string, std::string> eng_spn = {
        {"dog", "perro"}, {"cat", "gato"}, {"horse", "caballo"}
    };

    // Read a value by key, like eng_spn['dog'] in Python.
    std::cout << eng_spn["dog"] << "\n";

    // Add a new pair.
    eng_spn["cow"] = "waca";
    // Change the value for an existing key.
    eng_spn["cow"] = "vaca";
    // Delete a pair (removes both the key and its value).
    eng_spn.erase("horse");

    // Print every pair. The order is not guaranteed for unordered_map.
    for (const auto& kv : eng_spn){
        std::cout << kv.first << " -> " << kv.second << "\n";
    }
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>


### A few things to remember, matching the Python summary you saw:

- ### Create a map with `std::unordered_map<KeyType, ValueType> name;`. You name both types. Python does not make you do this.
- ### Read or write a value with `name[key]`.
- ### Add a new pair, or change an existing one, with `name[key] = value`.
- ### Delete a pair with `name.erase(key)`. This is the C++ version of Python's `del`.
- ### Go through every pair with a range-based `for`, using `kv.first` for the key and `kv.second` for the value.

In [ ]:
////mapLookup.cpp
// Example 8b - Looking up a missing key: Python raises an error, C++ does not.
//%CXXFLAGS -std=c++11
#include <iostream>
#include <unordered_map>
#include <string>

int main(){
    std::unordered_map<std::string, std::string> eng_spn = {
        {"dog", "perro"}, {"cat", "gato"}
    };

    // In Python, eng_spn['bird'] would raise a KeyError, so the notes used try/except.
    // In C++, eng_spn["bird"] does NOT raise anything. It silently INSERTS "bird"
    // with the default value (an empty string) and returns that.
    std::cout << "[" << eng_spn["bird"] << "]\n";
    std::cout << "size is now " << eng_spn.size() << "\n";

    // So you cannot use eng_spn[key] to test whether a key is present.
    // Even a test like this inserts, because the [] still runs:
    //     if (eng_spn["fish"] == "") ...

    // To test for a key without inserting, use count(key) (returns 0 or 1)
    // or find(key) (returns an iterator, equal to end() if the key is absent).
    if (eng_spn.count("cat")) std::cout << "cat -> " << eng_spn["cat"] << "\n";
    if (eng_spn.count("fish") == 0) std::cout << "fish is not in the map\n";

    auto it = eng_spn.find("dog");
    if (it != eng_spn.end()) std::cout << "found " << it->first << " -> " << it->second << "\n";
    return 0;
}


<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

### Think of `count` and `find` as C++'s version of Python's `in`. They answer "is this key here?" without changing the map.

### Iterating a map uses the same range-based `for` pattern from section 4. Each element is a `std::pair`, with `.first` the key and `.second` the value. In Python, `for k in d` gives you the keys in no particular order. C++ is similar: with `std::unordered_map` the pairs come out in no particular order either. (If you used `std::map` instead, they would come out in sorted key order. More on that at the end of the section.)

In [ ]:
////mapIterate.cpp
// Example 8c - Going through every pair in a map
//%CXXFLAGS -std=c++11
#include <iostream>
#include <unordered_map>
#include <string>

int main(){
    // Like the Python vote_count = {'Alice':3, 'Bob':1, 'Carol':1}
    std::unordered_map<std::string, int> vote_count = {
        {"Alice", 3}, {"Bob", 1}, {"Carol", 1}
    };

    for (const auto& kv : vote_count){
        std::cout << kv.first << " got " << kv.second << " votes\n";
    }
    // The order of the lines is not guaranteed. You might see Alice first,
    // or Bob first, and so on. If you need the keys sorted, use std::map instead.
    return 0;
}


<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

### **You try!** (Example 8d) Write a function `wordCount` that takes a `std::vector<std::string>` and returns a `std::unordered_map<std::string, int>` of how many times each word appears. Try it on `{"apple", "banana", "apple", "cherry", "banana", "apple"}`. You should get `apple -> 3`, `banana -> 2`, `cherry -> 1`.

### Hint: this is the case where the auto-insert from Example 8b is actually useful. `++counts[word]` creates the entry with value 0 if the word is new, then increments it. So one line handles both the "first time I've seen this word" case and the "seen it before" case.

In [ ]:
////wordCount.cpp
// Example 8d - wordCount with std::unordered_map
// (You write this!)
//%CXXFLAGS -std=c++11
#include <iostream>
#include <unordered_map>
#include <string>
#include <vector>

int main(){
    // TODO: write your solution here, then test it.
    return 0;
}

### **`unordered_map` vs `map`: which one?**

### `std::unordered_map` is a hash table. Looking up, inserting, or erasing a key takes constant time on average, written O(1). The cost is that the pairs have no order: when you iterate, they come out in whatever order the hash table happens to store them. This is what Python's `dict` is.

### `std::map` is a balanced binary search tree (a red-black tree, if you want the name). That structure is the reason the keys stay sorted. It also means lookup, insert, and erase take O(log n) time instead of O(1). That is a little slower, but still very fast.

### A simple guideline: use `std::unordered_map` unless you specifically need the keys in sorted order, in which case use `std::map`. Most of the time you just want "give me the value for this key," and that is the `unordered_map` case.

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 9. Memoized Fibonacci with `static`

### Now we'll combine three things in one example: maps, recursion, and a new keyword called `static`.

### Recall the Fibonacci sequence: 0, 1, 1, 2, 3, 5, 8, 13, 21, ... where each number is the sum of the previous two. Defined recursively:

- `fib(0) = 0`
- `fib(1) = 1`
- `fib(n) = fib(n-1) + fib(n-2)` for `n >= 2`

### The recursive definition translates directly into code.

In [ ]:
////fibNaive.cpp
// Example 9a - Naive recursive Fibonacci
// Try fib(40). On most machines this takes several seconds.
//%CXXFLAGS -std=c++11
#include <iostream>

int fib(int n){
    if (n < 2) return n;
    return fib(n-1) + fib(n-2);
}

int main(){
    std::cout << fib(10) << "\n";   // 55, instant
    std::cout << fib(30) << "\n";   // 832040, noticeable pause
    // std::cout << fib(40) << "\n";   // works but slow (uncomment to feel it)
    return 0;
}

<br><br><br><br><br>
<br><br><br><br><br>


### **Why so slow?** Trace the call tree for `fib(5)`:

```
                        fib(5)
                      /        \
                  fib(4)        fib(3)
                  /    \        /    \
              fib(3)  fib(2)  fib(2)  fib(1)
              /   \   /  \    /  \
          fib(2) fib(1) ...
```

### `fib(3)` gets computed **twice**. `fib(2)` gets computed **three times**. By the time you call `fib(40)`, you're recomputing `fib(2)` over **a hundred million times**. The work doubles roughly every step you go up: that's **exponential time**.

### **The fix: remember answers we've already computed.** This trick is called **memoization**. We'll store each computed value in a `std::unordered_map<int, int>`. The next time we need it, we just look it up instead of recomputing.

### One wrinkle: if the map is a regular local variable inside `fib`, it gets created fresh on every recursive call, which defeats the purpose. We need *one map shared across every call*. That's where the `static` keyword comes in.

### A **`static` local variable** inside a function is initialized **once** (the first time the function is called) and **persists** for the lifetime of the program. Unlike a normal local, it isn't destroyed when the function returns.

In [ ]:
////fibMemo.cpp
// Example 9b - Memoized fib with static std::unordered_map. fib(40) is now instant.
//%CXXFLAGS -std=c++11
#include <iostream>
#include <unordered_map>

int fib(int n){
    static std::unordered_map<int, int> memo;   // initialized once, persists across all calls
    if (n < 2) return n;
    if (memo.count(n)) return memo[n];  // already computed? return it.
    memo[n] = fib(n-1) + fib(n-2);      // otherwise compute, store, return.
    return memo[n];
}

int main(){
    std::cout << fib(10) << "\n";   // 55
    std::cout << fib(40) << "\n";   // 102334155, now instant
    std::cout << fib(50) << "\n";   // 12586269025, still instant
    return 0;
}

### Three concepts in one example:

### 1. **`std::unordered_map`** stores the cached answers, keyed by `n`.
### 2. **Recursion** is the same as before. `fib` calls itself.
### 3. **`static`** makes the local map survive across all the recursive calls, instead of being recreated each time.

### The technique has a name: **memoization**, a form of **dynamic programming**. You'll see more of it in an algorithms class.

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 10. Where to Go From Here

### The C++ we covered is enough to read most real code and write small programs of your own. Here's a short list of topics that go beyond what we did, in case you want to keep learning. Each of these has solid free tutorials online (cppreference.com is the canonical reference; learncpp.com is friendlier).

### **RAII** (Resource Acquisition Is Initialization). The pattern we've been using all semester has a name. When a constructor acquires a resource (memory, a file handle, a network connection, a lock) and the destructor releases it, that's RAII. It's the main reason C++ doesn't need `try/finally` blocks the way Java and Python do. The destructor takes care of cleanup automatically when the object goes out of scope.

### **Smart pointers** (`std::unique_ptr`, `std::shared_ptr`, in `<memory>`). These are wrapper classes around raw pointers that call `delete` for you in their destructors. Modern C++ rarely uses `new`/`delete` directly. Instead, you wrap the pointer in a `unique_ptr` and let RAII clean up.

### **More of the STL.** We touched `vector`, `list`, `unordered_map`. There's also `set`, `map`, `unordered_set`, `deque`, `queue`, `stack`, `array`, `bitset`. And the `<algorithm>` header has dozens of functions beyond `find` and `count`: `sort`, `reverse`, `transform`, `accumulate`, `any_of`, `all_of`, `for_each`, and more.

### **Templates beyond what we did.** Template specialization (a custom version for a specific type), variadic templates (templates that take any number of type parameters), and concepts (C++20, a way to constrain what types a template will accept). Also worth looking at: `std::variant` and `std::optional` for type-safe alternatives to pointers and null.

### **Multiple inheritance.** A class can inherit from more than one base. Powerful, but introduces tricky cases like the "diamond problem" (what happens when two parents share a grandparent). Some languages (Java, C#) forbid it; C++ allows it.

### **Virtual functions and polymorphism.** We did inheritance without `virtual`. Here's what it adds: mark a method `virtual` in the base class and override it in each derived class, and then a call through a base-class pointer or reference runs the *derived* version, chosen at runtime from the actual object type. That's what lets you keep, say, a `std::vector<Shape*>` of mixed `Circle`s and `Rectangle`s and call `area()` on each without caring which is which. Related topics: pure virtual functions (`= 0`) and abstract classes, virtual destructors (almost always needed once you have virtual functions), and `dynamic_cast` for safe downcasting.

### **Move semantics** (`std::move`, `&&` rvalue references). Lets objects efficiently transfer ownership of their resources instead of deep-copying. This is what makes modern C++ fast. It's a chunky topic but well worth learning if you keep going.